In [37]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib
import os

# Load Dataset

In [38]:
# df = pd.read_csv("../data/sintetis/dataset_irigasi.csv")
df = pd.read_csv("../data/dataset_irigasi.csv")
print(f"Dataset: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

Dataset: 1507 baris, 4 kolom


,soil_moisture,air_temperature,air_humidity,irrigation_action
0,27.79,30.5,74.6,0
1,28.00,30.4,76.6,0
2,28.00,30.4,76.6,0
3,27.79,30.3,77.1,0
4,27.79,30.3,76.5,0


# Pisahkan Fitur & Label

In [39]:
FEATURES = ["soil_moisture", "air_temperature", "air_humidity"]

X = df[FEATURES].values
y = df["irrigation_action"].values

print(f"Fitur: {FEATURES}")
print(f"Label 0 (tidak siram): {(y==0).sum()}")
print(f"Label 1 (siram):       {(y==1).sum()}")
print(f"Rasio siram: {y.mean()*100:.1f}%")

Fitur: ['soil_moisture', 'air_temperature', 'air_humidity']
Label 0 (tidak siram): 827
Label 1 (siram):       680
Rasio siram: 45.1%


In [40]:
if len(set(y)) < 2:
    print("⚠️ BAHAYA: cuma 1 kelas! Data belum variatif, model gak bisa dilatih.")
else:
    print(f"OK — {len(set(y))} kelas. Lanjut.")

OK — 2 kelas. Lanjut.


# Split Train/Test

In [41]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42  # removed stratify=y
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")


Train: 1205, Test: 302


# Training Random Forest

In [42]:
model = RandomForestClassifier(
    n_estimators=15,     
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
print("Training selesai.")

Training selesai.


# Cross Validation

In [43]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1")
print(f"CV F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

CV F1: 0.9944 (+/- 0.0035)


# Evaluasi di Test Set

In [44]:
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1: {f1_score(y_test, y_pred):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
print(f"\n{classification_report(y_test, y_pred, target_names=['Tidak siram', 'Siram'])}")


#emang kalo gapake labelling bisa belajar pola alami, tapi kan petani nyiramnya sesuai jadwal doang, jadi langkah paling efisien untuk labelling ya mempelajari sumber sumber yang relefan, petani kan menyiram fix aja ngikut jam tidak pakai sensor, jadi kaya percuma aja saya ngambil data label dan ngajarin model saya hanya berdasarkan kebiasaan petani yang cuman ngikutin jam, bukan data mikroklimat yang relevan, jadi modelnya bakal belajar pola yang salah. Jadi saya harus labelling data sesuai dengan kondisi mikroklimat yang relevan.

Accuracy: 0.9934
F1: 0.9933

Confusion Matrix:
[[151   2]
 [  0 149]]

              precision    recall  f1-score   support

 Tidak siram       1.00      0.99      0.99       153
       Siram       0.99      1.00      0.99       149

    accuracy                           0.99       302
   macro avg       0.99      0.99      0.99       302
weighted avg       0.99      0.99      0.99       302



# Feature importance

In [45]:
importances = model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
print("Feature Importance:")
for idx in sorted_idx:
    bar = "█" * int(importances[idx] * 50)
    print(f"  {FEATURES[idx]:18s} {importances[idx]:.4f}  {bar}")

Feature Importance:
  soil_moisture      0.9167  █████████████████████████████████████████████
  air_humidity       0.0445  ██
  air_temperature    0.0389  █


# Test Prediksi Manual

In [46]:


# --- Test case: (deskripsi, soil_moisture, air_temp, air_humidity, harapan) ---
# Variabel EC dan sensor mati/spike jenuh dihapus karena model RF sudah bersih dari data rusak
kasus_uji = [
    ("Sebelum siram PAGI (acuan petani)",   11.0, 23.9, 71.9,  1),  # kering pagi -> siram
    ("Sebelum siram JAM 11 (panas+kering)", 25.0, 32.9, 42.0,  1),  # panas -> ambang naik -> siram
    ("Baru disiram (basah)",                36.5,  9.4, 24.7,  0),  # basah -> jangan
    ("Zona aman siang normal",              22.0, 28.0, 60.0,  0),  # cukup -> jangan
    ("Malam sangat lembap, agak kering",    12.5, 16.0, 90.0,  1),  # ah>85 -> ambang turun -> siram
]

print(f"{'Deskripsi':<38}{'sm':>6}  {'hasil AI':>8}{'harap':>8}  status")
print("-" * 75)

lulus = 0
for desk, sm, at, ah, harap in kasus_uji:
    # Susun input jadi format array 2D yang diminta Random Forest
    X_sample = np.array([[sm, at, ah]])
    
    # AI melakukan tebakan!
    hasil = int(model.predict(X_sample)[0])
    
    ok = "OK" if hasil == harap else "XX GAGAL"
    if hasil == harap:
        lulus += 1
        
    print(f"{desk:<38}{sm:>6}  {hasil:>8}{harap:>8}  {ok}")

print("-" * 75)
print(f"Lulus: {lulus}/{len(kasus_uji)}")

Deskripsi                                 sm  hasil AI   harap  status
---------------------------------------------------------------------------
Sebelum siram PAGI (acuan petani)       11.0         1       1  OK
Sebelum siram JAM 11 (panas+kering)     25.0         0       1  XX GAGAL
Baru disiram (basah)                    36.5         0       0  OK
Zona aman siang normal                  22.0         0       0  OK
Malam sangat lembap, agak kering        12.5         0       1  XX GAGAL
---------------------------------------------------------------------------
Lulus: 3/5


# Simpan Model

In [47]:
import os
os.makedirs("models", exist_ok=True)

joblib.dump(model, "models/rf_irigasi.joblib")
print("Model tersimpan: models/rf_irigasi.joblib")

Model tersimpan: models/rf_irigasi.joblib


# Convert ke esp

In [49]:
from micromlgen import port

c_code = port(model)
with open("model_irigasi.h", "w") as f: 
    f.write(c_code)
print("Tersimpan: model_irigasi.h")
print(f"Ukuran: {len(c_code)} char, {len(c_code.encode('utf-8'))} bytes")

Tersimpan: model_irigasi.h
Ukuran: 63993 char, 63993 bytes
